# 4.2 Evaluate XGBoost vs benchmark (run in SageMaker Studio)

Batch transform on the test split, metrics (macro-F1 headline), comparison with the benchmark and a confusion matrix. **Test scores are optimistic**: test windows come from subjects seen in training (chronological split, spec B). The last cell is a manual approval step - run it only if you are satisfied.

In [ ]:
import sys, pathlib
ROOT = pathlib.Path.cwd()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
import io, json
import boto3, numpy as np, pandas as pd
from src.pamap2.config import load_config, s3_uri
from src.pamap2.schema import ACTIVITY_NAMES
from src.pamap2 import evaluation as E
cfg = load_config()
session = boto3.Session(region_name=cfg["region"])
print("Caller:", session.client("sts").get_caller_identity()["Arn"])
s3 = session.client("s3")
SPLITS = s3_uri(cfg, "splits")
REPORTS = s3_uri(cfg, "models", "reports/")

def split_key(uri):
    b, k = uri[len("s3://"):].split("/", 1)
    return b, k
def read_parquet(name):
    b, k = split_key(SPLITS + name)
    return pd.read_parquet(io.BytesIO(s3.get_object(Bucket=b, Key=k)["Body"].read()))
def read_json(uri):
    b, k = split_key(uri)
    return json.loads(s3.get_object(Bucket=b, Key=k)["Body"].read())
def write_json(uri, obj):
    b, k = split_key(uri)
    s3.put_object(Bucket=b, Key=k, Body=json.dumps(obj, indent=2).encode())
    print("Wrote", uri)

FEATURES = read_json(SPLITS + "feature_columns.json")
LABEL_MAP = E.load_label_map(read_json(SPLITS + "label_map.json"))   # class index -> activity_id
K = len(LABEL_MAP)
CLASSES = list(range(K))
def class_name(i):
    return ACTIVITY_NAMES[LABEL_MAP[int(i)]]
print(K, "classes;", len(FEATURES), "features")


In [ ]:
import sagemaker
sm = session.client("sagemaker")
sm_session = sagemaker.Session(boto_session=session)
import time
from sagemaker.transformer import Transformer

def upload_features_csv(df, key):
    """Header-less, feature-only CSV in feature_columns.json order."""
    E.check_feature_columns(df, FEATURES)
    body = df[FEATURES].to_csv(header=False, index=False).encode()
    uri = s3_uri(cfg, "models", key)
    b, k = split_key(uri)
    s3.put_object(Bucket=b, Key=k, Body=body)
    return uri

def run_transform(model_name, input_uri, output_prefix, join=False):
    """Batch transform on ml.m5.large x1; waits for completion. Nothing stays running afterwards."""
    t = Transformer(model_name=model_name, instance_count=1, instance_type="ml.m5.large",
                    strategy="MultiRecord", assemble_with="Line", accept="text/csv",
                    output_path=output_prefix, sagemaker_session=sm_session)
    t.transform(data=input_uri, content_type="text/csv", split_type="Line",
                join_source="Input" if join else "None", wait=True, logs=False)
    return output_prefix + input_uri.rsplit("/", 1)[1] + ".out"

def read_probs(out_uri, k):
    b, kk = split_key(out_uri)
    return E.parse_probs(s3.get_object(Bucket=b, Key=kk)["Body"].read(), k)


## Create a temporary SageMaker model from the trained artifact

In [ ]:
train_info = read_json(REPORTS + "xgboost_training.json")
model_name = f"{cfg['prefix']}-eval-{int(time.time())}"
sm.create_model(ModelName=model_name, ExecutionRoleArn=cfg["role_arn"],
                PrimaryContainer={"Image": train_info["image_uri"], "ModelDataUrl": train_info["model_data"]})
test = read_parquet("test.parquet")
test_uri = upload_features_csv(test, "transform-input/test/features.csv")
print(len(test), "test rows ->", test_uri)

## Batch transform (billable: one ml.m5.large job, shuts down on completion)

In [ ]:
try:
    out_uri = run_transform(model_name, test_uri, s3_uri(cfg, "models", "transform-output/test/"))
    probs = read_probs(out_uri, K)
    assert len(probs) == len(test), (len(probs), len(test))
    pred = E.probs_to_labels(probs, K)
    metrics = E.classification_metrics(test["label"], pred, CLASSES)
finally:
    sm.delete_model(ModelName=model_name)   # only the temporary model created above

## Benchmark vs XGBoost

In [ ]:
bench = read_json(REPORTS + "benchmark.json")["test_tree"]
cmp = E.compare(bench, metrics)
print(f"HEADLINE macro-F1 (test): benchmark={bench['macro_f1']:.4f}  xgboost={metrics['macro_f1']:.4f}  better={cmp['better']}")
display(cmp["table"])
display(pd.DataFrame({class_name(k): v for k, v in metrics["per_class"].items()}).T)

In [ ]:
import matplotlib.pyplot as plt
cm = np.array(metrics["confusion_matrix"])
cmn = cm / np.maximum(cm.sum(axis=1, keepdims=True), 1)
fig, ax = plt.subplots(figsize=(9, 8))
im = ax.imshow(cmn, cmap="Blues", vmin=0, vmax=1)
names = [class_name(i) for i in CLASSES]
ax.set_xticks(range(K)); ax.set_xticklabels(names, rotation=90)
ax.set_yticks(range(K)); ax.set_yticklabels(names)
ax.set_xlabel("Predicted"); ax.set_ylabel("True"); ax.set_title("XGBoost test confusion matrix (row-normalised)")
fig.colorbar(im); plt.tight_layout(); plt.show()

## Save report

In [ ]:
write_json(REPORTS + "xgboost_eval.json", {**metrics, "label_map": {str(k): v for k, v in LABEL_MAP.items()},
    "benchmark_macro_f1": bench["macro_f1"], "better_than_benchmark": cmp["better"],
    "model_package_arn": train_info["model_package_arn"]})
sm.update_model_package(ModelPackageArn=train_info["model_package_arn"], CustomerMetadataProperties={
    "test_macro_f1": f"{metrics['macro_f1']:.6f}", "test_accuracy": f"{metrics['accuracy']:.6f}",
    "benchmark_macro_f1": f"{bench['macro_f1']:.6f}", "eval_report": REPORTS + "xgboost_eval.json",
    **{k.replace(":", "_"): f"{v:.6f}" for k, v in train_info["final_metrics"].items()}})

## MANUAL APPROVAL - do NOT run unless you are satisfied with the comparison above

There is no scripted gate. Running the next cell marks the model package `Approved`, which lets `5.0_deploy` use it.

In [ ]:
# MANUAL STEP: run only after reviewing the benchmark comparison.
if not cmp["better"]:
    print("!" * 70 + chr(10) + "WARNING: XGBoost did NOT beat the benchmark on test macro-F1. Approving anyway (no auto-block)." + chr(10) + "!" * 70)
sm.update_model_package(ModelPackageArn=train_info["model_package_arn"], ModelApprovalStatus="Approved",
                        ApprovalDescription="Approved manually after benchmark comparison in 4.2")
print("Approved:", train_info["model_package_arn"])